In [1]:
# Imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
# Vision Mamba block (Zhu et al. 2024) -- Mamba's selective SSM run in
# both directions (forward + backward), then combined. Lets any token
# influence any other token regardless of position.

class VisionMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2, dt_rank=None, conv_kernel=3, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand      # internal working width
        self.d_state = d_state               # size of hidden "memory" per channel
        self.dt_rank = dt_rank or max(d_model // 16, 4)

        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)  # -> main branch x, gate z

        # Local conv before the scan (cheap neighbour mixing)
        self.conv1d_fwd = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=conv_kernel,
                                     padding=conv_kernel - 1, groups=self.d_inner)
        self.conv1d_bwd = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=conv_kernel,
                                     padding=conv_kernel - 1, groups=self.d_inner)

        # delta, B, C computed fresh per token -- this is the "selective" part
        self.x_proj_fwd = nn.Linear(self.d_inner, self.dt_rank + d_state * 2, bias=False)
        self.x_proj_bwd = nn.Linear(self.d_inner, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj_fwd = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        self.dt_proj_bwd = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # A: fixed decay rate, learned once (not per-token)
        A_fwd = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        A_bwd = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log_fwd = nn.Parameter(torch.log(A_fwd))
        self.A_log_bwd = nn.Parameter(torch.log(A_bwd))
        self.D_fwd = nn.Parameter(torch.ones(self.d_inner))  # skip weight
        self.D_bwd = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model)
        self.dropout = nn.Dropout(dropout)

    def _scan(self, x_in, conv1d, x_proj, dt_proj, A_log, D):
        """One direction of the scan. x_in: (B, L, d_inner).
        deltaA/deltaB_x are (B, L, d_inner, d_state) -- memory scales with L."""
        B_, L, _ = x_in.shape
        x_in = conv1d(x_in.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_in = F.silu(x_in)

        x_dbl = x_proj(x_in)
        delta, Bp, Cp = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(dt_proj(delta))          # per-token step size

        A = -torch.exp(A_log)
        deltaA = torch.exp(delta.unsqueeze(-1) * A)                          # state decay this step
        deltaB_x = delta.unsqueeze(-1) * Bp.unsqueeze(2) * x_in.unsqueeze(-1) # new info written in

        h = torch.zeros(B_, self.d_inner, self.d_state, device=x_in.device, dtype=x_in.dtype)
        ys = []
        for t in range(L):                          # sequential recurrence across tokens
            h = deltaA[:, t] * h + deltaB_x[:, t]     # update hidden state
            y_t = torch.einsum('bdn,bn->bd', h, Cp[:, t])  # read output from state
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        y = y + x_in * D                             # skip connection
        return y

    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        y_fwd = self._scan(x_in, self.conv1d_fwd, self.x_proj_fwd, self.dt_proj_fwd,
                            self.A_log_fwd, self.D_fwd)

        x_in_rev = torch.flip(x_in, dims=[1])         # reverse sequence for backward pass
        y_bwd_rev = self._scan(x_in_rev, self.conv1d_bwd, self.x_proj_bwd, self.dt_proj_bwd,
                                self.A_log_bwd, self.D_bwd)
        y_bwd = torch.flip(y_bwd_rev, dims=[1])        # flip back to original order

        y = y_fwd + y_bwd                             # combine both directions
        y = y * F.silu(z)                             # gate

        out = self.out_proj(y)
        out = self.dropout(out)
        return out + residual

In [3]:
# Splits each 64^3 ROI into non-overlapping 8^3 patches, converts each
# patch to a token via 3D conv (stride = kernel = patch_size). All 6 ROIs'
# patches are concatenated into one sequence. 6 * 8*8*8 = 3072 tokens total.

class ROIPatchEmbed(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64):
        super().__init__()
        self.n_rois = n_rois
        self.grid_size = roi_size // patch_size            # 8 patches per side
        self.patches_per_roi = self.grid_size ** 3          # 512
        self.n_tokens = self.patches_per_roi * n_rois        # 3072

        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens, d_model) * 0.02)  # retains patch location

    def forward(self, rois):
        # rois: (B, n_rois, 1, 64, 64, 64)
        B, N = rois.shape[0], rois.shape[1]
        all_tokens = []
        for i in range(N):
            patches = self.patch_conv(rois[:, i])            # (B, d_model, 8, 8, 8)
            patches = patches.flatten(2).transpose(1, 2)       # (B, 512, d_model)
            all_tokens.append(patches)
        tokens = torch.cat(all_tokens, dim=1)                  # (B, 3072, d_model)
        return tokens + self.pos_embed

In [4]:
# Patch embed -> stacked Vim blocks -> mean pool

class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, d_state=16, expand=2, dropout=0.3):
        super().__init__()
        self.patch_embed = ROIPatchEmbed(n_rois, roi_size, patch_size, d_model)
        self.mamba_layers = nn.ModuleList([
            VisionMambaBlock(d_model, d_state=d_state, expand=expand, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, rois):
        tokens = self.patch_embed(rois)          # (B, 3072, d_model)
        for layer in self.mamba_layers:
            tokens = layer(tokens)
        tokens = self.norm(tokens)
        return tokens.mean(dim=1)                 # mean pool -> (B, d_model)

In [5]:
class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, expand=2, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model,
                                         n_layers, d_state, expand, dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        return self.classifier(self.branch(rois))

In [6]:
class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, expand=2, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model,
                                             n_layers, d_state, expand, dropout)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model,
                                             n_layers, d_state, expand, dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        mri_features = self.mri_branch(mri_rois)
        pet_features = self.pet_branch(pet_rois)
        fused = torch.cat([mri_features, pet_features], dim=1)  # late fusion
        return self.classifier(fused)

In [7]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

# Same stratified split used across the whole pipeline (random_state=42)
X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))


class ROIDataset(Dataset):
    """Single-modality dataset (MRI-only or PET-only)."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.load(f"{self.cache_dir}/{key}_{version}.npy").astype(np.float32)
        return torch.tensor(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long)


class MultimodalROIDataset(Dataset):
    """Pairs matching MRI and PET aug files per subject/seed."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy").astype(np.float32)
        pet_rois = np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy").astype(np.float32)
        return (torch.tensor(mri_rois).unsqueeze(1), torch.tensor(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long))

In [8]:
# Dataset loading - MRI Only example

train_dataset = ROIDataset(X_train, y_train, MRI_CACHE_AUG, is_mri=True, is_train=True)
val_dataset   = ROIDataset(X_val,   y_val,   MRI_CACHE_AUG, is_mri=True, is_train=False)
test_dataset  = ROIDataset(X_test,  y_test,  MRI_CACHE_AUG, is_mri=True, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=0)

In [9]:
# Sanity check: one forward+backward pass, small model, small batch.
# Confirms it runs on your GPU and shows actual memory use before you
# commit to a full 5-seed training run.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats()

model = VisionMambaModel(d_model=32, n_layers=1).to(device)  # start small
x = torch.randn(2, 6, 1, 64, 64, 64).to(device)               # batch of 2

out = model(x)
loss = out.sum()
loss.backward()

print(f"Output shape: {out.shape}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Output shape: torch.Size([2, 2])
Peak GPU memory: 0.31 GB


In [10]:
# Now test real settings:
# d_model=64 (proposal-scale), n_layers=2, batch_size=8 (existing convention)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = VisionMambaModel(d_model=64, n_layers=2).to(device)
x = torch.randn(8, 6, 1, 64, 64, 64).to(device)   # real batch size

out = model(x)
loss = out.sum()
loss.backward()

print(f"Output shape: {out.shape}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Output shape: torch.Size([8, 2])
Peak GPU memory: 3.76 GB


In [11]:
# Multimodel Memory Test
# (two full branches), so this is where OOM is most likely to actually happen

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

mm_model = MultimodalVisionMambaModel(d_model=64, n_layers=2).to(device)
mri_x = torch.randn(8, 6, 1, 64, 64, 64).to(device)
pet_x = torch.randn(8, 6, 1, 64, 64, 64).to(device)

out = mm_model(mri_x, pet_x)
loss = out.sum()
loss.backward()

print(f"Output shape: {out.shape}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Output shape: torch.Size([8, 2])
Peak GPU memory: 6.73 GB
